In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import joblib, os

df = pd.read_csv('../data/clean_matches.csv')

# Supprimer les lignes sans durée
df2 = df.dropna(subset=['duration_minutes', 'winner']).copy()
print("Shape après nettoyage:", df2.shape)

Shape après nettoyage: (636, 16)


In [2]:
le_cat   = LabelEncoder()
le_round = LabelEncoder()
le_court = LabelEncoder()
le_src   = LabelEncoder()

df2['category_enc']  = le_cat.fit_transform(df2['category'])
df2['round_name_enc']= le_round.fit_transform(df2['round_name'])
df2['court_enc']     = le_court.fit_transform(df2['court'].fillna('Unknown'))
df2['source_enc']    = le_src.fit_transform(df2['competition_source'])
df2['winner_enc']    = (df2['winner'] == 'team_1').astype(int)
df2['played_at']     = pd.to_datetime(df2['played_at'], errors='coerce')
df2['month']         = df2['played_at'].dt.month.fillna(0).astype(int)

FEATURES = ['category_enc', 'round', 'round_name_enc', 'index',
            'court_enc', 'source_enc', 'month', 'winner_enc']

X = df2[FEATURES]
y = df2['duration_minutes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (508, 8) | Test: (128, 8)


In [3]:
models = {
    "Ridge":         Ridge(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost":       XGBRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
    print(f"\n=== {name} ===")
    print(f"MAE:  {mae:.2f} min")
    print(f"RMSE: {rmse:.2f} min")
    print(f"R²:   {r2:.4f}")


=== Ridge ===
MAE:  24.33 min
RMSE: 29.50 min
R²:   -0.0009

=== Random Forest ===
MAE:  26.27 min
RMSE: 32.05 min
R²:   -0.1814

=== XGBoost ===
MAE:  26.87 min
RMSE: 34.17 min
R²:   -0.3427


In [4]:
os.makedirs('../models', exist_ok=True)
joblib.dump(models["Random Forest"], '../models/rf_regressor_matches.pkl')
joblib.dump(models["XGBoost"],       '../models/xgb_regressor_matches.pkl')
print("✅ Modèles régression sauvegardés !")

✅ Modèles régression sauvegardés !
